# Issue Index Generator

Generate a JSON index file for all issues in a given provider's OCR collection.
This tool discovers all issues in a provider directory and creates a structured index
organized by media_alias → year → month → issue records.

The output index can be used as input file for the impresso text import pipeline.

## Import Required Libraries

In [28]:
import json
import os
import re
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import logging
from tqdm import tqdm
from datetime import date, datetime
import pandas as pd
from ast import literal_eval
from tqdm import tqdm
import ijson

from impresso_essentials.utils import ALL_MEDIA
from text_preparation.utils import edition_num_to_code
# Register tqdm with pandas to enable progress_apply
tqdm.pandas()

## Configuration

Configure the provider collection parameters and output settings.
This changes for each provider's specific data situation

In [2]:
DATA_DIR = "../text_preparation/data"

## SUB case

In [5]:
# Provider name and media aliases mapping
PROVIDER_NAME = "SUB"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../text_preparation/data/issue_indices/issue_index.sub.json
Using SUB-specific directory structure discovery


### Load Provider Collection Data

Discover all OCR files in the provider directory structure.

In [6]:
dir_to_alias_sub = {
    "Hamburger_Echo": "hamb_echo",
    "Hamburger_Volksblatt": "hamb_volksbl",
    "Der_Hamburger_Beobachter": "hamb_beob",
    "Neue_hamburgische_Blaetter": "neu_hamb_bl",
    "Bergedorfer_Eisenbahn-Zeitung": "berg_eisb_z",
    "Norddeutsche_Zeitung": "norddt_ztg",
    "Hamburgischer_Correspondent": "hamb_corr",
    "Harburger_Anzeigen": "harb_anz",
    "Bergedorfer_Wochenblatt_zum_Nutzen_und_zur_Unterhaltung": "berg_wbl_nu",
    "Harburger_Anzeigen_und_Nachrichten": "harb_anz_nachr",
    "Altonaischer_Mercurius": "alt_merc",
    "Der_Hamburger_Beobachter_und_das_Archiv_Wissenschaften_und_Kuenste": "hamb_beob_arch",
    "Hamburgischer_Correspondent_und_Hamburgische_Boersen-Halle": "hamb_corr_boers",
    "Bergedorfer_Zeitung_und_Anzeiger": "berg_ztg_anz",
    "Eisenbahnzeitung": "eisb_ztg",
    "Relations-Courier": "rel_cour",
    "Hamburger_Tageblatt": "hamb_tagbl",
    "Stats-_und_gelehrte_Zeitung_des_hollsteinischen_unpartheyischen_Correspondenten": "stats_gel_holl",
    "Staats-_und_gelehrte_Zeitung_des_Hamburgischen_unpartheyischen_Correspondenten": "staats_gel_hamb",
    "Zuerst-bekandte_Schiffbecker_Stats-_und_gelehrte_Zeitung_des_hollsteinischen_unpartheyischen_Correspondenten": "zuerst_schiffb",
    "Gerichtszeitung": "gerichts_ztg",
    "Reichs-Post-Reuter": "reichs_post_r",
    "Billstedter_Zeitung": "billst_ztg",
    "Buergerzeitung": "buerger_ztg",
    "Bergedorfer_Zeitung": "berg_ztg",
    "Altonaer_Mercur": "alt_mercur",
    "Hamburger_Volkszeitung": "hamb_volksz",
    "Hamburger_Fremdenblatt": "hamb_fremdbl",
    "Privilegirter_Hollsteinischer_Avisen-Correspondente_durch_Europa_und_andere_Theile_der_Welt": "priv_holl_avis",
    "Hamburg-Altonaer_Volksblatt": "hamb_alt_volksbl",
    "Bergedorfer_Wochenblatt_und_Eisenbahn-Zeitung": "berg_wbl_eisb",
    "Hansische_Warte": "hans_warte",
    "Hamburger_Relations-Courier": "hamb_rel_cour",
    "Volksblatt_fuer_Harburg_Wilhelmsburg_und_Umgegend": "volksbl_harb",
    "Hamburgischer_Correspondent_und_neue_hamburgische_Boersen-Halle": "hamb_corr_neu"
}

In [5]:
any(v in ALL_MEDIA for v in dir_to_alias_sub.values() if v!='hamb_echo')

False

#### Define a sorting lambda function to assign edition letters 

In [6]:
# Edition sorting key: splits on '-', sorts by number first, then by edition type
# Returns (number, edition_priority) tuple
sort_edition = lambda e: (
    int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] else 999,
    {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1}.get(e.split('-')[-1] if '-' in e else e, 2)
)

# Test
test_editions = ["Ausgabe", "M1-Morgenausgabe", "A2-Abendausgabe", "M3-Morgenausgabe", "A1-Abendausgabe"]
print("Sorting with lambda:")
all_day_editions = sorted(test_editions, key=sort_edition)
for edition in test_editions:
    edition_l = chr(96 + all_day_editions.index(edition)+1)
    print(f"  {edition} → {sort_edition(edition)} --> {edition_l}")

Sorting with lambda:
  Ausgabe → (999, -1) --> e
  M1-Morgenausgabe → (1, 0) --> a
  A2-Abendausgabe → (2, 1) --> c
  M3-Morgenausgabe → (3, 0) --> d
  A1-Abendausgabe → (1, 1) --> b


### aggregate all in a function which creates and writes to disk the issue index

#### define a list of image format extensions

In [7]:
# sort the image formats from most desirable to least
possible_img_formats = [".jp2", ".tif", ".tiff", ".jpg", ".jpeg", ".pdf", ".png"]

In [ ]:
def get_issue_metadata_info(prov_base_dir, alias, issue_dir_path, base_dir: str = BASE_DATA_PATH, debug:bool=False):

    try:
        # extract the relative path to the issue
        parts = issue_dir_path.rstrip("/").split("/")
        year, month, day = parts[-4:-1]
        edition_name = parts[-1].split('/')[0]
            
        sort_editions = lambda e: (
            # there is a special case for Hamburger_Fremdenblatt/1942/08/19/: Abendausgabe  Auslandsausgabe  AV-Auslandsausgabe  Morgenausgabe  Spätausgabe
            int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] and e[1] != 'V' else 999,
            {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1, "Spätausgabe": 2, "Auslandsausgabe": 3}.get(e.split('-')[-1] if '-' in e else e, 2)
        )
        # For directories without explicit edition prefix
        # This will be resolved later when grouping by day
        day_dir = '/'.join(parts[:-1])
        all_day_editions = sorted([d for d in os.listdir(day_dir) if os.path.isdir(os.path.join(day_dir, d))], key=sort_editions)
        edition = chr(96 + all_day_editions.index(edition_name)+1)
        

        # IDENTIFY THE SUBDIR OF THE IMAGES
        non_xml_files = {'': [f for f in os.listdir(issue_dir_path) if not f.endswith('.xml')]}
        subdirs = [d for d in os.listdir(issue_dir_path) if os.path.isdir(os.path.join(prov_base_dir, d))]

        #if len(non_xml_files['']) == 0:
        # there are no other files than xmls check if there are subdirs which contain some
        for subdir in subdirs:
            non_xml_files[subdir] = [f for f in os.listdir(os.path.join(issue_dir_path, subdir)) if not f.endswith('.xml')]

        # IDENTIFY THE EXTENSION OF THE IMAGES FILES
        imgs_dir = []
        img_ext = []
        missing_image_files = None
        num_xml_files = len(list(Path(issue_dir_path).rglob('*.xml')))
        for subdir, files in non_xml_files.items():
            #if not imgs_dir and not img_ext:
                #print(f"ext: {ext}")
                #print(f"sum(ext in f for f in non_xml_files): {sum(ext in f for f in non_xml_files)+1}")
                #print(f"len(Path(issue_dir_path).rglob('*.xml'))+1: {len(list(Path(issue_dir_path).rglob('*.xml')))}")
            for ext in possible_img_formats:
                files_with_ext = [f for f in files if ext in f]
                if len(files_with_ext)+1 == num_xml_files:
                    # the extensions are sorted, the first one to match is the one of choice
                    imgs_dir.append(subdir)
                    img_ext.append(ext)
                    break
                elif debug:
                    print(f"WARNING!! subdir {issue_dir_path} and extension {ext} - found {len(files_with_ext)} image files, but there are {num_xml_files} xml files")
                if len(files_with_ext)!=0:
                    imgs_dir.append(subdir)
                    img_ext.append(ext)
                    # finding the exact page numbers of the images present
                    existing_image_files = [int(f.split('.')[0]) for f in files_with_ext if '.' in f and not f.startswith('.')]
                    missing_image_files = [i for i in range(1, num_xml_files) if i not in existing_image_files]
                    print(f"WARNING!! subdir {issue_dir_path} and extension {ext} - missing image files for pages {missing_image_files} - keeping note of it.")

        if len(imgs_dir)>1 or len(img_ext)>1:
            print(f"WARNING!! THERE MIGHT BE MULTIPLE IMAGE FORMATS; THE CHOSEN ONE WILL BE {img_ext[0]} PRESENT IN SUBDIR {imgs_dir[0]}, BUT THERE NEEDS TO BE AN XML CHECK!!")
        if len(imgs_dir)==0 or len(img_ext)==0:
            print(f"WARNING!! FOR PATH {issue_dir_path} WE ARE LIKELY MISSING ALL IMAGES; as img_ext={img_ext} and imgs_dir={imgs_dir}, THIS SHOULD BE CHECKED BY HAND, SETTING None.")
            imgs_dir.append(None)
            img_ext.append(None)
            

        # save the metadata for this issue
        issue_meta_info = {
            "day": day,
            "edition": edition,
            "local_path": [issue_dir_path.replace(base_dir, "")[1:]],
            "imgs_subdir": imgs_dir[0],
            "imgs_ext": img_ext[0]
        }

        if missing_image_files:
            issue_meta_info['missing_image_files'] = missing_image_files

        if debug:
            print(f"Adding issue {alias}-{year}-{month}-{day}-{edition} to the index: {issue_meta_info}")

        return year, month, issue_meta_info

    except Exception as e:
        print(f"Issue at path {issue_dir_path} presented an exception: {e}")
        raise e

In [15]:
def find_issues_w_glob(prov_base_dir, title_dirname, alias, base_dir: str = BASE_DATA_PATH, debug:bool=False) -> List[Dict]:
    """
    Discover SUB issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]

    USING RGLOB
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """
    alias_issues = {}

    all_mets_files = [str(p) for p in Path(os.path.join(prov_base_dir, title_dirname)).rglob("*/PPN*.xml")]
    print(f"Found {len(all_mets_files)} mets files for {alias}, sorting them by date...")
    # sort all issue files to write years and days in the correct order
    sorted_mets = sorted(all_mets_files, key=lambda x: date(int(x.rstrip("/").split("/")[-5]), int(x.rstrip("/").split("/")[-4]), int(x.rstrip("/").split("/")[-3])))

    print(f"Done sorting the mets files by date, now starting creating the index...")

    # look for all the mets files, which have a "PPN" in their filename -> this means the issue dir is also found
    for filepath in tqdm(sorted_mets):
        
        # extract the relative path to the issue
        issue_dir_path = os.path.dirname(filepath)

        year, month, issue_meta_info = get_issue_metadata_info(prov_base_dir, alias, issue_dir_path)

        # save the created metadata into the list of issues for this alias
        if year not in alias_issues:
            alias_issues[year] = {month: [issue_meta_info]}
        elif month not in alias_issues[year]:
            alias_issues[year][month] = [issue_meta_info]
        else:
            alias_issues[year][month].append(issue_meta_info)

    return alias_issues



def find_issues_w_walk(prov_base_dir, title_dirname, alias, base_dir: str = BASE_DATA_PATH, debug:bool=False) -> List[Dict]:
    """
    Discover SUB issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]

    USING OS.WALK
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """

    alias_issues = {}

    for issue_dir_path,dirs,files in tqdm(os.walk(os.path.join(prov_base_dir, title_dirname))):
        # only consider directories which contain files -> issue directories
        if files:

            if len([f for f in files if f.endswith('.xml')])==1:
                print(f"Warning! There is only one xml file in dir {issue_dir_path}, skipping this issue")
                continue

            year, month, issue_meta_info = get_issue_metadata_info(prov_base_dir, alias, issue_dir_path)
        
            # save the created metadata into the list of issues for this alias
            if year not in alias_issues:
                alias_issues[year] = {month: [issue_meta_info]}
            elif month not in alias_issues[year]:
                alias_issues[year][month] = [issue_meta_info]
            else:
                alias_issues[year][month].append(issue_meta_info)


    return alias_issues

In [16]:
def detect_issues_sub(out_path, base_dir: str = BASE_DATA_PATH, prov: str = PROVIDER_NAME, dir_to_alias_sub=dir_to_alias_sub, write_file:bool=True, debug:bool=False, resume=True, use_walk=True) -> List[Dict]:
    """
    Discover SUB newspaper issues using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]
    
    Issues are identified by the presence of METS XML files containing "PPN".
    
    Args:
        root_path: Root directory of the SUB collection
        
    Returns:
        List of issue metadata dictionaries with keys:
        {alias, year, month, day, edition, path, mets_file}
    """
    # First, list alias directories in base_dir for the provider
    prov_base_dir = os.path.join(base_dir, prov)

    try:
        title_dirs = [d for d in os.listdir(prov_base_dir) if os.path.isdir(os.path.join(prov_base_dir, d))]
    except OSError as e:
        print(f"Failed to list base directory {prov_base_dir}: {e}")
        return []

    if resume:
        with open(out_path, "r") as fin:
            all_issues = json.load(fin)
            print(f"The following aliases were already processed and will be skipped: {all_issues.keys()}")
    else:
        all_issues = {}

    for title_dirname, alias in dir_to_alias_sub.items():

        if alias in all_issues:
            print(f"Alias {alias} is already present in the index and will be skipped.")
            continue
        #elif alias == "hamb_corr":
        #    print(f"Skipping hamb_corr for now as it was taking too much time.")
        #    continue
        else:
            print(f"Processing to find the issues of alias {alias}...")

        if use_walk:    
            alias_issues = find_issues_w_walk(prov_base_dir, title_dirname, alias, base_dir, debug)
        else:
            alias_issues = find_issues_w_glob(prov_base_dir, title_dirname, alias, base_dir, debug)


        print(f"Adding {sum(len(d) for m in alias_issues.values() for d in m.values())} issues for alias {alias} to the total list of aliases")
        all_issues[alias] = alias_issues

        if write_file:
            with open(out_path, "w") as fout:
                json.dump(all_issues, fout, indent=4)

    return all_issues

In [ ]:
all_sub_issues = detect_issues_sub(OUTPUT_FILE)

Now sort all the results by year, month, day and edition letter to ease the use of the index.

In [7]:
with open(OUTPUT_FILE, "r") as fin:
    all_issues = json.load(fin)

In [12]:
all_sorted_issues = {}
duplicated_issues = {}
for alias, year_dicts in tqdm(all_issues.items()):
    all_sorted_issues[alias] = {}
    sorted_years = sorted(year_dicts.keys(), key=lambda x: int(x))
    for year in sorted_years:
    #for year, month_dicts in year_dicts.items():
        all_sorted_issues[alias][year] = {}
        sorted_months = sorted(year_dicts[year], key=lambda x: int(x))
        for month in sorted_months:
        #for month, days in month_dicts.items():
            sorted_issues = sorted(year_dicts[year][month], key=lambda x: int(x["day"]))
            if any(i['edition']==j['edition'] for i in sorted_issues for j in sorted_issues if i!=j and i['day']==j['day']):
                print(f"Warning! duplicated editions in the issues of {alias}-{year}-{month}: {sorted_issues}")
                duplicated_issues[{alias}] = [year, month]

            # now save the newly sorted issues
            all_sorted_issues[alias][year][month] = sorted_issues

  0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 35/35 [00:01<00:00, 22.62it/s]


In [14]:

with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_sorted_issues, fout, indent=4)

## RTS Case

In [2]:


# Provider name and media aliases mapping
PROVIDER_NAME = "RTS"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../../../data/issue_indices/issue_index.rts.json
Using SUB-specific directory structure discovery


### Read the processed metadata file, and filter out shows without audio files 

(also write this filtered version to disk)

In [ ]:
metadata_prep_filename = f"{DATA_DIR}/sample_data/RTS/unique_programs_metadata.rts.csv"

rts_metadata_df = pd.read_csv(metadata_prep_filename)
print(len(rts_metadata_df))
rts_metadata_df.head()

First remove the unused columns in this context.

The issue index is only used as a pointer to the data for each issue. 
Another dict will be created to store the metadata necessary for the canonical ingestion

In [ ]:
index_cols = ['alias', 'date_str', 'stripped_OID', 'stt_filename', 'final_mp3_filename']
df_for_index = rts_metadata_df[index_cols].copy()

df_for_index.head()

### Find duplicated values

In [17]:
duplicated = df_for_index[df_for_index.duplicated()]
print(f"There are {len(duplicated)} duplicated values in this data!, in the aliases {set(duplicated.alias.values)}")

There are 64 duplicated values in this data!, in the aliases {'ombres_eco', 'mag_eco'}


In [23]:
no_dupli = df_for_index.drop_duplicates()

# sort them by date
no_dupli['date_dt'] = pd.to_datetime(no_dupli['date_str'], format="%d/%m/%Y")
no_dupli = no_dupli.sort_values(by=['alias', 'date_dt'])

print(f"There are {len(no_dupli)} non-duplicated records.")

There are 26161 non-duplicated records.


In [ ]:
no_dupli

### Create the issue index file

In [ ]:
index_dict = no_dupli.to_dict('index')
index_dict

In [26]:
def get_edition(ed_num):
    if ed_num <= 26: 
        return chr(96+ed_num)
    # only works for editions > 26
    second_letter = ed_num%26
    first_letter = int(ed_num/26)
    if second_letter == 0:
        first_letter -= 1
        second_letter = 26
    #print(f"first_letter={first_letter}, second_letter={second_letter}")
    return chr(96+first_letter)+chr(96+second_letter)

In [ ]:
# check that it works
for i in range(25, 100):
    print(f"num={i}, ed:{get_edition(i)}")

In [ ]:
#rts_base_path = os.path.join(BASE_DATA_PATH, PROVIDER_NAME)
all_rts_issues = {alias: {} for alias in df_for_index['alias'].drop_duplicates().values}
# initialize the same way
num_eds_per_day = {alias: {} for alias in df_for_index['alias'].drop_duplicates().values}

for idx, row_dict in index_dict.items():

    day, month, year = row_dict['date_str'].split('/')

    alias = row_dict['alias']

    ## defining the edition
    if row_dict['date_str'] in num_eds_per_day[alias]:
        # there are already some issues for that day, increase by one the value and deduct the correct edition
        num_eds_per_day[alias][row_dict['date_str']] += 1
        # if there are more than 26 editions for a given day, the edition lettes should combine (aa, ab,...)
        issue_ed = get_edition(num_eds_per_day[alias][row_dict['date_str']])
    else:
        # if no issue already exists for that alias on that day, init the value 
        num_eds_per_day[alias][row_dict['date_str']] = 1
        issue_ed = 'a' 

    issue_dict = {
        "day": day,
        "edition": issue_ed,
        # the local path is the one of the MP3 file 
        "local_path": [os.path.join(PROVIDER_NAME, alias, "audio", row_dict['final_mp3_filename'])],
        "ext": ".mp3",
        "stripped_OID": row_dict['stripped_OID'],
        "xml_filepath": os.path.join(PROVIDER_NAME, alias, "stt", row_dict['stt_filename']),
    }

    # set the issue dict in the overall dict
    if year in all_rts_issues[alias]:
        if month in all_rts_issues[alias][year]:
            all_rts_issues[alias][year][month].append(issue_dict)
        else:
            all_rts_issues[alias][year][month] = [issue_dict]
    else:
        all_rts_issues[alias][year] = {month: [issue_dict]}


all_rts_issues

Save the resulting dict to disk

In [32]:
with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_rts_issues, fout, indent=4)

#### Check the cases where there is no valid file

for each of these files, we will try to find if there is any file which matches the OID

In [ ]:
last_alias = None
for row in no_existing_mp3s.itertuples():
    print(row.alias, row.stripped_OID, row.spt_filenames, row.mp3_filenames)
    if row.alias!=last_alias:
        # update the values if we changed alias
        audios_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "audio")
        stt_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "stt")
        last_alias = row.alias
        print(f"Changed values of audios_dir to {audios_dir} and stt_dir to {stt_dir}. Listing their contents")
        all_audios = os.listdir(audios_dir)
        all_stt = os.listdir(stt_dir)

    #matching_stt = row.stripped_OID
    stt_file = f"{row.stripped_OID}_STT.xml"
    print(f"{row.alias}: {stt_file in all_stt} - stt_file={stt_file}")

    support_filenames = [f for f in literal_eval(row.spt_filenames) if f]
    all_valid_audios = []
    for f in support_filenames:
        if f and '.wav' in f:
            print(f"removing the .wav in {f}")
            f = f.replace(".wav", '')
        valid_audios = [audio for audio in all_audios if f in audio]
        print(f"filename {f} corresponds to audio {valid_audios}!")
        all_valid_audios.extend(valid_audios)
    
    oid_also_in = [audio for audio in all_valid_audios if row.stripped_OID.lower() in audio]
    if len(all_valid_audios)>1:
        print(f"{row.alias}: There are more than 1 audio file which matches: {all_valid_audios}")
    elif len(all_valid_audios)==1:
        print(f"{row.alias}: audio_file={all_valid_audios[0]}")
    else:
        print(f"{row.alias}: No valid audio file")

## INA case

In [21]:
# Provider name and media aliases mapping
PROVIDER_NAME = "INA"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original/{PROVIDER_NAME}/Data"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"


print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")

Provider path: /mnt/project_impresso/original/INA/Data
Output file: ../text_preparation/data/issue_indices/issue_index.ina.json


### Read in the preprocessed file

In [ ]:
all_issues_df_path = f"{DATA_DIR}/sample_data/INA/all_issues_v2.csv"
df_for_index = pd.read_csv(all_issues_df_path, index_col=0)
df_for_index

#### Generating the issue index

In [16]:
# create a list of dicts view for the issues
index_dict = df_for_index.to_dict('index')
# obtain the list of aliases
ina_aliases = list(df_for_index['issue_id'].apply(lambda x: x.split('-')[0]).drop_duplicates().values)
len(ina_aliases)

44

In [ ]:
all_ina_issues = {alias: {} for alias in ina_aliases}

for idx, row_dict in index_dict.items():

    # the issue edition disambiguation was already done in the metadata preprocessing.
    alias, year, month, day, edition = row_dict['issue_id'].split('-')

    issue_dict = {
        "day": day,
        "edition": edition,
        # the local path is the one of the MP3 file 
        "local_path": literal_eval(row_dict['local_path']),
        "ext":  row_dict['extension'],
        "notice_id": row_dict['notice'],
        "xml_filepath": literal_eval(row_dict['xml_filepath']),
    }

    # there is one issue which had the wrong filename in the provided metadata
    if row_dict["issue_id"] == "TrParis-1953-12-10-a":
        issue_dict["local_path"] = [
            f.replace("_02_","_04_") for f in issue_dict["local_path"]
        ]
        issue_dict["xml_filepath"] = [
            f.replace("_02_","_04_") for f in issue_dict["xml_filepath"]
        ]

    # set the issue dict in the overall dict
    if year in all_ina_issues[alias]:
        if month in all_ina_issues[alias][year]:
            all_ina_issues[alias][year][month].append(issue_dict)
        else:
            all_ina_issues[alias][year][month] = [issue_dict]
    else:
        all_ina_issues[alias][year] = {month: [issue_dict]}


all_ina_issues

In [28]:
with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_ina_issues, fout, indent=4)

## KBR case

In [3]:
# Provider name and media aliases mapping
PROVIDER_NAME = "KBR"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original/{PROVIDER_NAME}"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"

# Pattern for extracting metadata from filenames
# Adjust these patterns based on your provider's naming convention
# Example pattern: 1924587_newspaper_actionfem_1927-10-15_01
FILENAME_PATTERN = r'.*?_newspaper_([a-z]+)_(\d{4})-(\d{2})-(\d{2})_(\d{2})'

# Example: For Olive format with dates in directory names
# FILENAME_PATTERN = r'(\d{4})/(\d{2})/(\d{2})/.*'

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Pattern: {FILENAME_PATTERN}")

Provider path: /mnt/project_impresso/original/KBR
Output file: ../text_preparation/data/issue_indices/issue_index.kbr.json
Pattern: .*?_newspaper_([a-z]+)_(\d{4})-(\d{2})-(\d{2})_(\d{2})


### Going through the KBR data to build the index

the KBR data is separated into two main directories: `16TB_disk` and `BelgicaPress`. Their contents are described inside [this issue](https://github.com/impresso/impresso-corpus-metadata/issues/53).

We know two types of title-level identifiers for KBR:
- The IDN (eg. `16841654` for `Avenir`)
- The internal KBR identifier (eg. `JB1199` for `Avenir`)

The two different subdirectory structure use different identifiers, but we have both for each title, so we will store both for each issue in the index.

In addition, issues have a unique 6-digit ID, and each page XML or jpeg file has a unique 8-digit ID. The issue ID is used in the issue-level metadata, and the page ID is used in the page-level metadata, so we will look into storing the issue IDs (or both) in the index. 

#### `16TB_disk`

This directory contains one subdirectory per title, named with an alias of the title (eg. `Avenir` for `L'Avenir`). Each of these subdirectories will be used as impresso alias for the title.
Under each of these title subdirectories, all the issue directories are flat, with the following naming convention: 
`[IDN]_[YYYYMMDD]_[6-digit number] > BE-KBR00_[IDN]_[YYYYMMDD]_00_00_00_0_01_[page number]_[8-digit unique number].[jpeg or xml]`

This structure is very simple and allows for easy creation of the issue index. 
However, there is a small caveat: 
- the `Charivari` folder --> the corresponding alias should be changed to `CharivariBE` since it's a duplicate of a BCUL alias
- any alias ending with `(deel)` --> this part is removed, and the troncated alias is used.

#### `BelgicaPress` 

It's contents are:
```
- 'BelgicaPress PDF'
- 'BelgicaPress TXT'
- 'BelgicaPress XML'
- 'JB685A'
- 'JB685_PDF'
- 'JB685_PDF_nomets'
- 'JB685_XML'
- 'JB685_XML_nomets'
```

Each of the subdirs BelgicaPress PDF, BelgicaPress TXT and BelgicaPress XML contain the directories named according to the KBR identifiers (in format JB[number]), with the following structure:
`BelgicaPress [EXT] > JB[number] > year > KB_JB[number]_[YYYY]-[MM]-[DD]-[edition]-[page number].[ext]`

- All images are in PDF format --> will need to be reformated
- All the xml/pdf files corresponding to an entire year for a given title are grouped together

Additionally, an 'Hors sélection' directory is also present in BelgicaPress PDF and BelgicaPress XML, containing the KBR identifiers for other titles, and following the same filestructure:
`'BelgicaPress [EXT]' > 'Hors sélection' > JB[number] > year > KB_JB[number]_[YYYY]-[MM]-[DD]-[edition]-[page number].[ext]`

`JB685A` will NOT be ingested from the BelgicaPress directory, as it is already present in the 16TB_disk directory.
`JB1023` will NOT be ingested because its PDF files are missing.

Finally, `JB685` is at a different directory level than the rest of the BelgicaPress directory, it's not exactly organized like the other titles, and the directories `nomets` should not be considered (we want to keep the mets). The directory structure for JB685_PDF and JB685_XML is flat, with the following naming convention:
`KB_JB685_[YYYY]-[MM]-[DD]-[edition]-[page number].[xml of pdf]` (and `KB_JB685_[YYYY]-[MM]-[DD]-[edition]-mets.xml`/`KB_JB685_[YYYY]-[MM]-[DD]-[edition]-.pdf` for the mets and compiled pdf files).

#### Decision

A good approach is to start with a dict, mapping each alias to it's IDN and KBR identifier (and possibly the directory litteral name), so that we can iterate more easily through the issues and create the index. 

### 1. Creating the issue index for the `16TB_disk` directory

In [56]:
# Mapping from the directory names in KBR/16TB_disk to the media alias and KBR identifiers
# idn = IDN (ID Notice), kbr_id = KBR internal identifier
dir_to_info_kbr_16tbdisk = {
    "Algemeennieuws": {"alias": "algemeennieuws", "idn": "16723367", "kbr_id": "JB1153"},
    "Amidelordre": {"alias": "amidelordre", "idn": "16716544", "kbr_id": "JB5"},
    "Avenir": {"alias": "avenir", "idn": "16841654", "kbr_id": "JB1199"},
    "Avondecho": {"alias": "avondecho", "idn": "17199990", "kbr_id": "JB483"},
    "Belge": {"alias": "belge", "idn": "16114263", "kbr_id": "JB14"},
    "Bienpublic": {"alias": "bienpublic", "idn": "16211508", "kbr_id": "JB432A"},
    "Brusselerzeitung": {"alias": "brusselerzeitung", "idn": "16777065", "kbr_id": "JB1158"},
    "Bruxellois": {"alias": "bruxellois", "idn": "16777935", "kbr_id": "JB28"},
    "Catholiquedespaysbas": {"alias": "catholiquedespaysbas", "idn": "15958072", "kbr_id": "JB31"},
    "Charivari": {"alias": "charivaribe", "idn": "17404053", "kbr_id": "JB32"},
    "Cineradio": {"alias": "cineradio", "idn": "12154874", "kbr_id": "B 3.405b"},
    "Courrierdafrique(deel)": {"alias": "courrierdafrique", "idn": "15402627", "kbr_id": "JB1140"},
    "Courrierpaysbas": {"alias": "courrierpaysbas", "idn": "16358009", "kbr_id": "JB54"},
    "Dedag": {"alias": "dedag", "idn": "17131445", "kbr_id": "JB1065"},
    "Echodeliege": {"alias": "echodeliege", "idn": "17136308", "kbr_id": "JB89"},
    "Gazetvanmechelen": {"alias": "gazetvanmechelen", "idn": "17151577", "kbr_id": "JB519"},
    "Hetlaatsteuur": {"alias": "hetlaatsteuur", "idn": "17145890", "kbr_id": "JB306"},
    "Independance": {"alias": "independance", "idn": "17213443", "kbr_id": "JB1236"},
    "Journaldenamur": {"alias": "journaldenamur", "idn": "17180654", "kbr_id": "JB1178"},
    "Journaldesflandres": {"alias": "journaldesflandres", "idn": "16126183", "kbr_id": "JB156"},
    "Journaldetournai": {"alias": "journaldetournai", "idn": "17180380", "kbr_id": "JB1176A"},
    "journaldeverviers": {"alias": "journaldeverviers", "idn": "17180934", "kbr_id": "JB1193"},
    "Journalduborinage": {"alias": "journalduborinage", "idn": "17180389", "kbr_id": "JB1176B"},
    "Journalducentre": {"alias": "journalducentre", "idn": "17180632", "kbr_id": "JB1177"},
    "LaBelgique": {"alias": "labelgique", "idn": "16770091", "kbr_id": "JB17"},
    "Lanterne": {"alias": "lanterne", "idn": "15425143", "kbr_id": "JB1228"},
    "Lynx": {"alias": "lynx", "idn": "17142621", "kbr_id": "JB182"},
    "Malmedyerkreisblatt": {"alias": "malmedyerkreisblatt", "idn": "17141952", "kbr_id": "JB173"},
    "Nieuwegids": {"alias": "nieuwegids", "idn": "15429590", "kbr_id": "JB1223B"},
    "Nieuwsblad(deel)": {"alias": "nieuwsblad", "idn": "17753860", "kbr_id": "JB657"},
    "Patriebelge": {"alias": "patriebelge", "idn": "17144109", "kbr_id": "JB229B"},
    #"Petitbleu(deel)": {"alias": "petitbleu", "idn": "17159796", "kbr_id": "JB685"}, # for separate, later ingest
    "Petitbleuexposition": {"alias": "petitbleuexposition", "idn": "17203915", "kbr_id": "JB685A"},
    "Peuplewallon": {"alias": "peuplewallon", "idn": "17144218", "kbr_id": "JB236"},
    "Presse": {"alias": "presse", "idn": "17159871", "kbr_id": "JB693"},
    "Regiondecharleroi": {"alias": "regiondecharleroi", "idn": "17144780", "kbr_id": "JB260"},
    "Rexjette": {"alias": "rexjette", "idn": "17177763", "kbr_id": "JB1080"},
    "Rhenanie": {"alias": "rhenanie", "idn": "17243242", "kbr_id": "JB667B"},
    "Telegraphe": {"alias": "telegraphe", "idn": "17145101", "kbr_id": "JB287"},
    "Vaderland": {"alias": "vaderland", "idn": "17145958", "kbr_id": "JB310"},
    "Vaderlandhet": {"alias": "vaderlandhet", "idn": "17145911", "kbr_id": "JB308"},
    "Vlaamscheland": {"alias": "vlaamscheland", "idn": "17180758", "kbr_id": "JB1183"},
    "Vlaanderenjongdietschland": {"alias": "vlaanderenjongdietschland", "idn": "17164908", "kbr_id": "JB779"},
    "Vlaemschbelgie": {"alias": "vlaemschbelgie", "idn": "17258557", "kbr_id": "JB317"},
    "Vlaemschebelgen": {"alias": "vlaemschebelgen", "idn": "17258570", "kbr_id": "JB318"},
}

In [26]:
datetime.strptime('19490401','%Y%m%d').day

1

In [58]:
disk_issues_kbr = {v['alias']: {} for k,v in dir_to_info_kbr_16tbdisk.items()}
# keep track of whether we ever find more than edition for an alias on a given date.
dailies_per_alias = {v['alias']: {} for k,v in dir_to_info_kbr_16tbdisk.items()}

disk_data_dir = os.path.join(BASE_DATA_PATH, "16TB_disk")
for dirname in os.listdir(disk_data_dir):

    if dirname == "Petitbleu(deel)":
        print(f"Skipping {disk_data_dir}, it will be processed later.")
        continue
    print(f"Preparing the issue index for dir: {disk_data_dir}")
    title_ids_dict = dir_to_info_kbr_16tbdisk[dirname]
    print(f" --> corresponding alias and IDs: {title_ids_dict}")
    alias = title_ids_dict['alias']

    for issue_dirname in os.listdir(os.path.join(disk_data_dir, dirname)):

        # extract info from the issue dirname
        alias_idn, str_date, issue_kbr_id = issue_dirname.split('_')

        if alias_idn!=title_ids_dict['idn']:
            print(f"Warning! The found IDN does not match the one we have for this title!! found IDN={alias_idn} - expected={title_ids_dict['idn']}") 

        if str_date not in dailies_per_alias[alias]:
            # the only issue for a given date - edition is a:
            edition = 'a'
            dailies_per_alias[alias][str_date] = [issue_dirname]
        elif issue_dirname not in dailies_per_alias[alias][str_date]:
            # ensure that this issue was not already considered.
            dailies_per_alias[alias][str_date].append(issue_dirname)
            edition = edition_num_to_code(len(dailies_per_alias[alias][str_date]))
        else:
            print(f"Warning!! The issue dirname {issue_dirname} was already present in the dailies for alias {alias}, skipping!! ")
            pass

        date_obj = datetime.strptime(str_date,'%Y%m%d')
        issue_dict = {
            "day": date_obj.day,
            "edition": edition,
            "local_path": os.path.join(PROVIDER_NAME, "16TB_disk", dirname, issue_dirname),
            "ext": 'jpeg',
            "issue_kbr_id": issue_kbr_id,
            "title_kbr_id": title_ids_dict['kbr_id'],
            "imgs_subdir": "",
        }

        # set the issue dict in the overall dict
        if date_obj.year in disk_issues_kbr[alias]:
            if date_obj.month in disk_issues_kbr[alias][date_obj.year]:
                disk_issues_kbr[alias][date_obj.year][date_obj.month].append(issue_dict)
            else:
                disk_issues_kbr[alias][date_obj.year][date_obj.month] = [issue_dict]
        else:
            disk_issues_kbr[alias][date_obj.year] = {date_obj.month: [issue_dict]}


disk_issues_kbr

Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'alias': 'lynx', 'idn': '17142621', 'kbr_id': 'JB182'}
Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'alias': 'rexjette', 'idn': '17177763', 'kbr_id': 'JB1080'}
Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'alias': 'vlaemschebelgen', 'idn': '17258570', 'kbr_id': 'JB318'}
Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'alias': 'presse', 'idn': '17159871', 'kbr_id': 'JB693'}
Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'alias': 'petitbleuexposition', 'idn': '17203915', 'kbr_id': 'JB685A'}
Preparing the issue index for dir: /mnt/project_impresso/original/KBR/16TB_disk
 --> corresponding alias and IDs: {'

{'algemeennieuws': {1944: {1: [{'day': 26,
     'edition': 'a',
     'local_path': 'KBR/16TB_disk/Algemeennieuws/16723367_19440126_667721',
     'ext': 'jpeg',
     'issue_kbr_id': '667721',
     'title_kbr_id': 'JB1153',
     'imgs_subdir': ''},
    {'day': 8,
     'edition': 'a',
     'local_path': 'KBR/16TB_disk/Algemeennieuws/16723367_19440108_667708',
     'ext': 'jpeg',
     'issue_kbr_id': '667708',
     'title_kbr_id': 'JB1153',
     'imgs_subdir': ''},
    {'day': 25,
     'edition': 'a',
     'local_path': 'KBR/16TB_disk/Algemeennieuws/16723367_19440125_667701',
     'ext': 'jpeg',
     'issue_kbr_id': '667701',
     'title_kbr_id': 'JB1153',
     'imgs_subdir': ''},
    {'day': 9,
     'edition': 'a',
     'local_path': 'KBR/16TB_disk/Algemeennieuws/16723367_19440109_667718',
     'ext': 'jpeg',
     'issue_kbr_id': '667718',
     'title_kbr_id': 'JB1153',
     'imgs_subdir': ''},
    {'day': 6,
     'edition': 'a',
     'local_path': 'KBR/16TB_disk/Algemeennieuws/16723367_1

### 2. Creating the issue index for the `BelgicaPress` directory

In [ ]:
# Mapping from the directory names in KBR/16TB_disk to the media alias and KBR identifiers
# idn = IDN (ID Notice), kbr_id = KBR internal identifier
dir_to_info_kbr_bp = {
    "JB1051": {"alias": "drapeauvoix", "idn": "16344089", "kbr_id": "JB1051"},
    "JB421": {"alias": "avenirluxembourg", "idn": "17149097", "kbr_id": "JB421"},
    "JB427": {"alias": "librebelgique", "idn": "16159138", "kbr_id": "JB427"},
    "JB555": {"alias": "independancebelge", "idn": "17229173", "kbr_id": "JB555"},
    "JB555A": {"alias": "independanceangl", "idn": "17240725", "kbr_id": "JB555A"},
    "JB567": {"alias": "journalbruxelles", "idn": "16411586", "kbr_id": "JB567"},
    "JB572": {"alias": "journalcharleroi", "idn": "17201736", "kbr_id": "JB572"},
    "JB638": {"alias": "meuse", "idn": "17159117", "kbr_id": "JB638"},
    "JB729": {"alias": "vingtiemesiecle", "idn": "16343920", "kbr_id": "JB729"},
    "JB773": {"alias": "versavenir", "idn": "15442494", "kbr_id": "JB773"},
    "JB837": {"alias": "peuple", "idn": "16155305", "kbr_id": "JB837"},
    "JB838": {"alias": "soir", "idn": "15438406", "kbr_id": "JB838"},
    "JB230": {"alias": "patriote", "idn": "16126160", "kbr_id": "JB230"},
    "JB258": {"alias": "reforme", "idn": "17144723", "kbr_id": "JB258"},
    "JB449": {"alias": "courrierdelescaut", "idn": "17113130", "kbr_id": "JB449"},
    "JB494": {"alias": "etoilebelge", "idn": "16365174", "kbr_id": "JB494"},
    "JB685": {"alias": "petitbleu", "idn": "17159796", "kbr_id": "JB685"},
    "JB835": {"alias": "derniereheure", "idn": "17171827", "kbr_id": "JB835"},
    "JB92": {"alias": "echoduparlement", "idn": "16344996", "kbr_id": "JB92"},
}

TODO check also that both the PDF and the XML files are present for each issue, and that the number of pages is the same in both files. If not, we will log a warning and skip the issue.

Note some are directly inside BelgicaPress, others inside BelgicaPress PDF/XML, and some are inside the 'Hors sélection' subdirectory. We will need to check all of these locations to find all the issues.

In [54]:
for dirname, vals in dir_to_info_kbr_bp.items():
    if vals['alias'] in disk_issues_kbr:
        print(f"{vals['alias']} present in the 16TB_disk data!")

petitbleu present in the 16TB_disk data!


### Special Cases

There are some special cases which appear in both base directories.

#### a. JB685: `{"alias": "petitbleu", "idn": "17159796", "kbr_id": "JB685"}`

- In BelgicaPress, we have 4 directories at the base level: `JB685_PDF   JB685_PDF_nomets   JB685_XML   JB685_XML_nomets`
- In 16TB_disk, we have: `"Petitbleu(deel)": {"alias": "petitbleu", "idn": "17159796", "kbr_id": "JB685"}`

There is an overlap in the data between the two directories, so we need to identify what is present in each subdirectory to make an informed decision.

In [45]:
# contents of the 16TB_disk
issues_per_year_jb685_disk = {k: sum(len(x) for x in v.values()) for k,v in disk_issues_kbr['petitbleu'].items()}
for k in sorted(issues_per_year_jb685_disk.keys(), key=lambda x: int(x)):
    print(f"{k}: {issues_per_year_jb685_disk[k]}")

1894: 231
1895: 353
1896: 339
1897: 345
1898: 316
1899: 347
1900: 352
1901: 356
1902: 351
1903: 348
1904: 179
1905: 358
1906: 359
1907: 368
1908: 356
1909: 137
1912: 20


In [46]:
bp_JB685_data_dir = os.path.join(BASE_DATA_PATH, "BelgicaPress/JB685_XML")

# check that there are never multiple editions for a given day
editions= set([f.split('_')[-1].split('-')[0] for f in os.listdir(bp_JB685_data_dir)])
print(f" present editions: {editions}")

issues_jb685_bp = set([f.split('_')[-2] for f in os.listdir(bp_JB685_data_dir)])
issues_per_year_jb685_bp = Counter(sorted([x.split('-')[0] for x in issues_jb685_bp], key=lambda x:int(x)))

for k in sorted(issues_per_year_jb685_bp.keys(), key=lambda x: int(x)):
    print(f"{k}: {issues_per_year_jb685_bp[k]}")

 present editions: {'01'}
1894: 230
1895: 347
1896: 336
1897: 344
1898: 311
1899: 347
1900: 352
1901: 356
1902: 351
1903: 174
1904: 179
1905: 358
1906: 359
1907: 364
1908: 356
1909: 360
1910: 347
1911: 344
1912: 156
1913: 329
1919: 149


### Petitbleu duplicates: `16TB_disk/Petitbleu(deel)` vs `BelgicaPress/JB685_XML`

**Comparison (MD5 on every page XML, 4,917 shared dates, 22,789 pages)**
- 22,717 pages (99.7%) are byte-identical; 4,902 of 4,917 dates are fully identical. The only difference is the file naming
  (`BE-KBR00_17159796_YYYYMMDD_…_NNNN_<id>.xml` vs `KB_JB685_YYYY-MM-DD_01-NNNN.xml`).
  Both are ABBYY FineReader Engine 11, ALTO v2, OCR produced 2018-06 to 2018-07.
- 14 dates differ (1896-05-05/06, 1898-02-13, 1898-06-26, 1900-01-21/22/23, 1900-03-14/15/25,
  1907-10-12, 1907-11-02, 1907-11-05, 1907-12-29). In these, `Petitbleu(deel)` holds an extra
  **tesseract 5.3.4** re-OCR (ALTO v3, no `processingDateTime`, folder IDs ~921330/940966, different
  page pixel dimensions, text similarity only ~0.2-0.4 with the ABBYY text). The ABBYY copy is still
  identical to JB685_XML. The Tesseract production date is not recorded; the 2025-05-03 file
  timestamps only reflect the disk copy.
- Page-set mismatches (not Tesseract-vs-ABBYY): 1898-02-13 page 6 exists only in the Tesseract folder
  `17159796_18980213_921330` (ABBYY folder `…_147073` has pages 1-5); 1909-05-17 pages 1-2 exist only in
  JB685_XML (`…_19090517_153012` lacks them).

**Decision**
1. Identical dates: JB685_XML is the canonical source (simple names, METS per issue); the Petitbleu copy is redundant, and `issue_dict['skip_if_duplicated'] = True` is set in the issue index to ensure we choose the correct source. Also the edition will be kept as `a` for the JB685_XML version found if there is already another issue with the same date and edition (hence form the 16TB disk).
2. The 14 differing dates: import the **ABBYY** version by default (consistent with the rest of the
   corpus). Keep the Tesseract version as a separate, explicitly flagged source and do **not** mix the
   two OCRs within an issue (different text and coordinate scales).
3. Pages missing on one side: take them from whichever folder has them, record the OCR engine per
   page, and check against the METS before relying on them (not yet verified).
   - 1898-02-13 -> keep the tesseract version (921330).
   - 1909-05-17 -> keep the JB685_XML version

DECISION: creating the issue index of Petitbleu separately from the rest, starting by the contents of BelgicaPress/JB685_XML, and then adding the Petitbleu(deel) issues which are not present in the BelgicaPress index, as well as the duplicates with explicit mentions not to ingest them. The importer will need to be aware of the `skip_if_duplicated` flag to avoid importing the Petitbleu(deel) issues which are already present in the BelgicaPress index.


To use later:
```
 petitbleu is almost completely duplicated into the BelgicaPress folder - which should be favored
        if alias_idn == '17159796':
            if issue_kbr_id == '921330':
                # special case where this specific version of an issue has 6 pages and the others only 5
                issue_dict['skip_if_duplicated'] = False
            else:
                issue_dict['skip_if_duplicated'] = True
```

In [9]:
bp_data_dir = os.path.join(BASE_DATA_PATH, "BelgicaPress/BelgicaPress PDF")

for root, dir, files in os.walk(bp_data_dir):
    if not files:
        print(root, dir, files)

/mnt/project_impresso/original/KBR/BelgicaPress/BelgicaPress PDF ['JB838', 'JB1051', 'JB638', 'JB837', 'JB555A', 'JB729', 'JB427', 'JB572', 'JB555', 'JB567', 'Hors sélection', 'JB421', 'JB773'] []
/mnt/project_impresso/original/KBR/BelgicaPress/BelgicaPress PDF/JB838 ['1923', '1960', '1898', '1933', '1942', '1957', '1901', '1950', '1892', '1895', '1900', '1943', '1931', '1951', '1964', '1959', '1887', '1919', '1944', '1912', '1965', '1946', '1914', '1928', '1889', '1924', '1945', '1932', '1913', '1956', '1893', '1935', '1929', '1967', '1952', '1888', '1907', '1934', '1966', '1911', '1903', '1958', '1930', '1961', '1947', '1896', '1890', '1939', '1904', '1941', '1910', '1955', '1927', '1922', '1938', '1891', '1937', '1954', '1920', '1962', '1905', '1940', '1894', '1963', '1968', '1953', '1926', '1948', '1906', '1921', '1969', '1909', '1970', '1949', '1902', '1908', '1897', '1936', '1925', '1918'] []


KeyboardInterrupt: 

From what we can see, there are clear overlaps between the two directories, and the `petitbleu` data is partly represented in both directories. 
I will store the indices from both directories and only ingest when necessary.
However, we will need to make the image copy process aware of the fact that not all issues should be ingested.

In [ ]:
bp_issues_kbr = {v['alias']: {} for k,v in dir_to_info_kbr_bp.items()}
# keep track of whether we ever find more than edition for an alias on a given date.
dailies_per_alias_bp = {v['alias']: {} for k,v in dir_to_info_kbr_bp.items()}

bp_data_dir = os.path.join(BASE_DATA_PATH, "BelgicaPress")

# first step - BelgicaPress XML etc
for dirname in os.listdir(disk_data_dir):

    print(f"Preparing the issue index for dir: {disk_data_dir}")
    title_ids_dict = dir_to_info_kbr_16tbdisk[dirname]
    print(f" --> corresponding alias and IDs: {title_ids_dict}")
    alias = title_ids_dict['alias']

    for issue_dirname in os.listdir(os.path.join(disk_data_dir, dirname)):

        # extract info from the issue dirname
        alias_idn, str_date, issue_kbr_id = issue_dirname.split('_')

        if alias_idn!=title_ids_dict['idn']:
            print(f"Warning! The found IDN does not match the one we have for this title!! found IDN={alias_idn} - expected={title_ids_dict['idn']}") 

        if str_date not in dailies_per_alias[alias]:
            # the only issue for a given date - edition is a:
            edition = 'a'
            dailies_per_alias[alias][str_date] = [issue_dirname]
        elif issue_dirname not in dailies_per_alias[alias][str_date]:
            # ensure that this issue was not already considered.
            dailies_per_alias[alias][str_date].append(issue_dirname)
            edition = edition_num_to_code(len(dailies_per_alias[alias][str_date]))
        else:
            print(f"Warning!! The issue dirname {issue_dirname} was already present in the dailies for alias {alias}, skipping!! ")
            pass

    date_obj = datetime.strptime(str_date,'%Y%m%d')
    issue_dict = {
        "day": date_obj.day,
        "edition": edition,
        "local_path": os.path.join(PROVIDER_NAME, "16TB_disk", dirname, issue_dirname),
        "ext": 'jpeg',
        "issue_kbr_id": issue_kbr_id,
        "title_kbr_id": title_ids_dict['kbr_id']
    }

    # set the issue dict in the overall dict
    if date_obj.year in disk_issues_kbr[alias]:
        if date_obj.month in disk_issues_kbr[alias][date_obj.year]:
            disk_issues_kbr[alias][date_obj.year][date_obj.month].append(issue_dict)
        else:
            disk_issues_kbr[alias][date_obj.year][date_obj.month] = [issue_dict]
    else:
        disk_issues_kbr[alias][date_obj.year] = {date_obj.month: [issue_dict]}


disk_issues_kbr

## BNF case

In [3]:
# Provider name and media aliases mapping
PROVIDER_NAME = "BNF"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}_new.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using BNF-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../text_preparation/data/issue_indices/issue_index.bnf_new.json
Using BNF-specific directory structure discovery


In [4]:
def find_bnf_issues_w_walk(prov_base_dir, alias, base_dir: str = BASE_DATA_PATH, skipped_issues:list|None=None) -> List[Dict]:
    """
    Discover BNF issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_letter]

    USING OS.WALK
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """

    alias_issues = {}
    alias_dir = os.path.join(prov_base_dir, alias)
    for issue_dir_path,dirs,files in tqdm(os.walk(alias_dir)):
        # only consider directories which contain files -> issue directories
        if alias=='oeuvre' and ('toc' in issue_dir_path or 'ocr' in issue_dir_path or (dirs and files)):
            # if we are working with oeuvre, some issue_dirs have the "old" format and should be skipped
            # this can be identified because the old format had a dir per issue which contains other directories
            skipped_issues.append(issue_dir_path)
            continue
        elif files:
            manifest_file = [f for f in files if f=='manifest.json']
            if len(manifest_file)==0:
                print(f"Warning! Manifest JSON file is missing in dir {issue_dir_path}, skipping this issue (dirs={dirs}, files={files})")
                continue
            elif len(manifest_file)>1:
                print(f"Warning! More than 1 Manifest JSON file in dir {issue_dir_path}, using the first one. (dirs={dirs}, files={files})")

            # the issue edition disambiguation was already done in the download
            #print(f"issue_dir_path.replace(alias_dir, ''): {issue_dir_path.replace(alias_dir, '')}")
            year, month, day, edition = issue_dir_path.replace(alias_dir, '')[1:].split('/')

            manifest_path = os.path.join(issue_dir_path, manifest_file[0])
            with open(manifest_path, 'rb') as f:
                id_url = next(ijson.items(f, 'id'))
        
            issue_dict = {
                "day": day,
                "edition": edition,
                # the local path is the one of the MP3 file 
                "local_path": [issue_dir_path.replace(base_dir, '')[1:]],
                "ark_id": id_url.split('/')[-2],
                "batch": "BNF_API_NEW"
            }
        
            # save the created metadata into the list of issues for this alias
            if year not in alias_issues:
                alias_issues[year] = {month: [issue_dict]}
            elif month not in alias_issues[year]:
                alias_issues[year][month] = [issue_dict]
            else:
                alias_issues[year][month].append(issue_dict)


    return alias_issues, skipped_issues

In [5]:
def detect_issues_bnf(out_path, base_dir: str = BASE_DATA_PATH, prov: str = PROVIDER_NAME, write_file:bool=True, debug:bool=False, resume=True, use_walk=True, aliases_to_override=None) -> List[Dict]:
    """
    Discover BNF newspaper issues using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]
    
    Issues are identified by the presence of METS XML files containing "PPN".
    
    Args:
        root_path: Root directory of the SUB collection
        
    Returns:
        List of issue metadata dictionaries with keys:
        {alias, year, month, day, edition, path, mets_file}
    """
    # the titles which were already shared with us for Impresso 1 have another internal organisation
    imp1_dirname_to_alias = {
        'Excelsior': 'excelsior',
        'La-Fronde': 'lafronde',
        'Marie-Claire': 'marieclaire',
        'Oeuvre': 'oeuvre', # oeuvre contains both old and new data - both need to be listed in issue_index
    }

    # First, list alias directories in base_dir for the provider
    prov_base_dir = os.path.join(base_dir, prov)

    try:
        title_dirs = [d for d in os.listdir(prov_base_dir) if os.path.isdir(os.path.join(prov_base_dir, d)) and '2020' not in d and not d.startswith('.')]
    except OSError as e:
        print(f"Failed to list base directory {prov_base_dir}: {e}")
        return []

    print(f"Found {len(title_dirs)} title directories: {title_dirs}")

    if resume and os.path.exists(out_path):
        with open(out_path, "r") as fin:
            all_issues = json.load(fin)
            print(f"The following aliases were already processed and will be skipped: {all_issues.keys()}")
    else:
        all_issues = {}

    oeuvre_skipped_issues = None
    if aliases_to_override is None:
        aliases_to_override = []

    for idx, alias in enumerate(title_dirs):

        if alias in imp1_dirname_to_alias:
            # replace with the correct alias
            alias = imp1_dirname_to_alias[alias]
        
        if alias in all_issues and alias not in aliases_to_override:
            print(f"{idx+1}/{len(title_dirs)} - Alias {alias} is already present in the index and will be skipped.")
            continue
        else:
            print(f"{idx+1}/{len(title_dirs)} - Processing to find the issues of alias {alias}...")

        if alias not in ['excelsior', 'lafronde', 'marieclaire']:    
            # skip the impresso 1 data for now, will add later
            if alias == 'oeuvre': 
                alias_issues, oeuvre_skipped_issues = find_bnf_issues_w_walk(prov_base_dir, alias, base_dir, [])
            else:
                alias_issues, _ = find_bnf_issues_w_walk(prov_base_dir, alias, base_dir, None)

        print(f"Adding {sum(len(d) for m in alias_issues.values() for d in m.values())} issues for alias {alias} to the total list of aliases")
        all_issues[alias] = alias_issues

        if write_file:
            with open(out_path, "w") as fout:
                json.dump(all_issues, fout, indent=4)

    return all_issues, oeuvre_skipped_issues

In [6]:
OUTPUT_FILE

'../text_preparation/data/issue_indices/issue_index.bnf_new.json'

In [10]:
aliases_to_override = ["revuecolan", "rhcf", "togocameroun"]
aliases_to_override = ["bsgecm",
"bcaf",
"actionfrancaise1899",
"actionfrancaise1908",
#"echoalger",
#"echoran",
#"humanite", # will be done in a second time
"ikdam",
"intransigeant",
"univers",
"lajustice",
"lapresse",
"canardenchaine",
"leconstitutionnel",
"figaro1826",
#"figaro1854", #
"figarosupl",
#"lepetitjournal",
#"letemps", # will be done in a second time
"revuecol"
]

In [11]:
all_sub_issues, skipped_issues = detect_issues_bnf(OUTPUT_FILE, aliases_to_override=aliases_to_override)

Found 129 title directories: ['jeuneeurope1930', 'letemps', 'quinzainecol', 'vsve', 'petitepresse', 'petitmarocain', 'bsgecm', 'lescolonies', 'pressecolill', 'lafrique1844', 'liberation', 'lecaucase', 'unionfrancaise', 'jeuneeuroperev', 'combat', 'franceyougoslavie', 'ikdam', 'democratiepacifique', 'terreeurope', 'europeor1919', 'gazettecoloniale', 'bocfo', 'nyherald', 'cridespeuples', 'etatsuniseurope', 'europedabord', 'europecoloniale', 'lepays', 'libertecol', 'lesbalkans', 'parisbalkans', 'demokratischeztg', 'depechetoulouse', 'progrescol', 'bulletinoffcol', 'revuecolan', 'Oeuvre', 'europedanubienne', 'laliberte', 'figarosupl', 'armeecoloniale', 'lacharente', 'elouma', 'chicagotribune', 'europeindcom', 'europefuture', 'nouvellestcheco', 'parismidi', 'europeartistique', 'lepeuple', 'notretemps', 'europefinanciere1', 'candide', 'francerussie', 'courrierlondres', 'bulletincol', 'mondecolill', 'lacroix', 'intransigeant', 'lecourrier', 'lapresse', 'vendredi', 'europenouvelle', 'europefin

26it [00:02,  9.36it/s]


Adding 7 issues for alias bsgecm to the total list of aliases
8/129 - Alias lescolonies is already present in the index and will be skipped.
9/129 - Alias pressecolill is already present in the index and will be skipped.
10/129 - Alias lafrique1844 is already present in the index and will be skipped.
11/129 - Alias liberation is already present in the index and will be skipped.
12/129 - Alias lecaucase is already present in the index and will be skipped.
13/129 - Alias unionfrancaise is already present in the index and will be skipped.
14/129 - Alias jeuneeuroperev is already present in the index and will be skipped.
15/129 - Alias combat is already present in the index and will be skipped.
16/129 - Alias franceyougoslavie is already present in the index and will be skipped.
17/129 - Processing to find the issues of alias ikdam...


538it [00:54,  9.95it/s]


Adding 224 issues for alias ikdam to the total list of aliases
18/129 - Alias democratiepacifique is already present in the index and will be skipped.
19/129 - Alias terreeurope is already present in the index and will be skipped.
20/129 - Alias europeor1919 is already present in the index and will be skipped.
21/129 - Alias gazettecoloniale is already present in the index and will be skipped.
22/129 - Alias bocfo is already present in the index and will be skipped.
23/129 - Alias nyherald is already present in the index and will be skipped.
24/129 - Alias cridespeuples is already present in the index and will be skipped.
25/129 - Alias etatsuniseurope is already present in the index and will be skipped.
26/129 - Alias europedabord is already present in the index and will be skipped.
27/129 - Alias europecoloniale is already present in the index and will be skipped.
28/129 - Alias lepays is already present in the index and will be skipped.
29/129 - Alias libertecol is already present i

5241it [08:07, 10.75it/s]


Adding 2814 issues for alias figarosupl to the total list of aliases
41/129 - Alias armeecoloniale is already present in the index and will be skipped.
42/129 - Alias lacharente is already present in the index and will be skipped.
43/129 - Alias elouma is already present in the index and will be skipped.
44/129 - Alias chicagotribune is already present in the index and will be skipped.
45/129 - Alias europeindcom is already present in the index and will be skipped.
46/129 - Alias europefuture is already present in the index and will be skipped.
47/129 - Alias nouvellestcheco is already present in the index and will be skipped.
48/129 - Alias parismidi is already present in the index and will be skipped.
49/129 - Alias europeartistique is already present in the index and will be skipped.
50/129 - Alias lepeuple is already present in the index and will be skipped.
51/129 - Alias notretemps is already present in the index and will be skipped.
52/129 - Alias europefinanciere1 is already pr

3202it [05:52,  9.07it/s]


KeyboardInterrupt: 